In [29]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
import operator

In [30]:
load_dotenv()

True

In [31]:
model = ChatOpenAI(model="gpt-4o-mini")

In [32]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Feedback on the essay")
    score: int = Field(description="Score out of 10")

In [33]:
structured_model = model.with_structured_output(EvaluationSchema)

In [34]:
essay = """The Indian economy has been on a growth trajectory for the past few decades, with significant contributions from various sectors. However, the COVID-19 pandemic has posed unprecedented challenges to the economy, leading to a contraction in GDP and a rise in unemployment. The government has implemented several measures to mitigate the impact of the pandemic, including fiscal stimulus packages and monetary policy interventions. As we look towards recovery, it is crucial to focus on sustainable growth, job creation, and addressing structural issues in the economy. The role of technology and innovation will also be pivotal in driving economic growth and ensuring resilience against future shocks."""

In [35]:
prompt = f"Please provide feedback and a score out of 10 for the following essay:\n\n{essay}"
structured_model.invoke(prompt)

EvaluationSchema(feedback="Your essay effectively outlines the challenges faced by the Indian economy due to the pandemic and briefly touches upon the government's response. It highlights important areas for future focus, including sustainable growth, job creation, and technological innovation. However, it could benefit from a more in-depth analysis of specific measures taken by the government or examples of sectors that have been particularly affected. Additionally, including statistics or data points could strengthen your arguments and provide more credibility. Overall, your essay is concise and clear, but it lacks depth and detail that would enhance the discussion.", score=6)

In [36]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    average_score: float

In [37]:
def evaluate_language(state: UPSCState) -> UPSCState:
    prompt = f"Evaluate the language and grammar of the following essay:\n\n {state['essay']}"
    output = structured_model.invoke(prompt)
    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

In [38]:
def evaluate_analysis(state: UPSCState) -> UPSCState:
    prompt = f"Evaluate the depth of analysis of the following essay:\n\n {state['essay']}"
    output = structured_model.invoke(prompt)
    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

In [39]:
def evaluate_thought(state: UPSCState) -> UPSCState:
    prompt = f"Evaluate the thought of the following essay:\n\n {state['essay']}"
    output = structured_model.invoke(prompt)
    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

In [40]:
def final_evaluation(state: UPSCState) -> UPSCState:
    prompt = f"Evaluate the {state['language_feedback']}, {state['analysis_feedback']} and {state['clarity_feedback']} of all these three feedback"

    overall_feedback = model.invoke(prompt)

    average_score = sum(state['individual_scores']) / len(state['individual_scores'])

    return {'overall_feedback': overall_feedback, 'average_score': average_score}
    

In [42]:
graph = StateGraph(UPSCState)

# nodes
graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

# edges
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)

workflow = graph.compile()

In [43]:
initial_state = {'essay': essay}

workflow.invoke(initial_state)

{'essay': 'The Indian economy has been on a growth trajectory for the past few decades, with significant contributions from various sectors. However, the COVID-19 pandemic has posed unprecedented challenges to the economy, leading to a contraction in GDP and a rise in unemployment. The government has implemented several measures to mitigate the impact of the pandemic, including fiscal stimulus packages and monetary policy interventions. As we look towards recovery, it is crucial to focus on sustainable growth, job creation, and addressing structural issues in the economy. The role of technology and innovation will also be pivotal in driving economic growth and ensuring resilience against future shocks.',
 'language_feedback': 'Overall, the essay demonstrates a good command of language and grammar. The sentences are structured clearly and convey the message effectively. The use of relevant economic terms is appropriate and contributes to the professional tone of the essay. There are no 